In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os, sys
import torch
import numpy as np
import pandas as pd 
import random
import json
import re

from pathlib import Path

# To set deterministic behaviour:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')

from mmengine.config import Config, DictAction
from mmengine.logging import print_log
from mmengine.registry import RUNNERS
from mmengine.runner import Runner
from mmdet.evaluation import DumpDetResults

from mmdet.utils import setup_cache_size_limit_of_dynamo


def set_seed(seed):
    # Set the seed for generating random numbers in PyTorch
    torch.manual_seed(seed)
    # If using GPUs, ensure that the random numbers are generated the same way
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    
    # Set the seed for generating random numbers in Python
    random.seed(seed)
    
    # Set the seed for generating random numbers in numpy
    np.random.seed(seed)
    
    # Ensure deterministic behavior by setting the flag
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optionally, set environment variables to ensure reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(42)

def init_cfg(folder):
    cfg = Config.fromfile(f'{folder}/vfnet_r18.py')
    return cfg

def extract_first_band(path):
    # Regular expression to extract the first number after the first 'b'
    match = re.search(r'perfect_b(\d+)', path)

    # Extracted value
    if match:
        first_b_number = match.group(1)
        return int(first_b_number)

def list_weights(folder):
    return [x for x in Path(folder).rglob('*.pth') if 'best' not in str(x)]
    
def set_NoiseTesting_config(weight_path, severity=0, corruption='gaussian', Seed=18):
    """
    Set the configuration for noise testing.
    Args:
        weight_path (str): The path to the weight file.
        severity (int, optional): The severity of the corruption. Defaults to 0.
        corruption (str, optional): The type of corruption. Defaults to 'gaussian'.
    Returns:
        Config: The modified configuration object.
    """
    

    CORRUPTION = corruption
    SEVERITY = severity
    IMG_SIZE = 2048 # Default
    
    if isinstance(weight_path, str) :
        cfgDir = Path(weight_path).parent
    else:
        cfgDir = weight_path.parent
    
    
    cfg = init_cfg(folder=f'{cfgDir}')
    band_sel = extract_first_band(weight_path.as_posix())
    assert band_sel is not None, f'Band not found in the weight path: {weight_path}'
    assert band_sel in [x for x in range(1,13,1)], f'Band not in the range [1,12]: {band_sel}'
    
    # Modify Config
    CHECKPOINT = weight_path
    cfg.load_from = f"{CHECKPOINT}"
    print(f'Loading from: {CHECKPOINT}')
    cfg.test_dataloader.dataset.pipeline = [{'type': 'SelBandLoader', 'to_float32': True, 'bands_list': [band_sel]},
                        dict(type='LoadAnnotations', with_bbox=True),
                        dict(keep_ratio=False, scale=(IMG_SIZE,IMG_SIZE,), type='Resize'),
                        dict(type='ImageCorruption', corruption=CORRUPTION, severity=SEVERITY), # 'gaussian', 
                        dict(
                            meta_keys=('img_path', 'img_id', 'seg_map_path', 
                                    'height', 'width', 'instances', 'sample_idx', 
                                    'img', 'img_shape', 'ori_shape', 'scale', 'scale_factor', 
                                    'keep_ratio', 'homography_matrix', 'gt_bboxes', 'gt_ignore_flags', 
                                    'gt_bboxes_labels'),
                            type='PackDetInputs'),
                    ]

    work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
    cfg.work_dir = work_dir
    
    # Hook for using the SIoU intestead of the IoU:
    cfg.test_evaluator.type = 'SIoUCocoMetric'
    return cfg

In [4]:
band_sel = 5
S = 0.1
NOISE = 'spike'

def get_weight_b(band_sel):
    weigths = list_weights(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b{band_sel}')
    assert len(weigths) > 0, 'No weights found'
    weight_path = weigths[0]
    return weight_path

weigths = list_weights(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b{band_sel}')
assert len(weigths) > 0, 'No weights found'
weight_path = weigths[0]
cfg = set_NoiseTesting_config(weight_path, severity=S, corruption=NOISE, Seed=18)

Loading from: /Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b5/18_BS_3_LR_0.0008_ME_30_OPT_SGD/epoch_25.pth


In [5]:
runner = RUNNERS.build(cfg)
work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
output_test_data = runner.test()

08/30 12:29:05 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 18
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda-11.4
    NVCC: Cuda compilation tools, release 11.4, V11.4.152
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/30 12:29:21 - mmengine - INFO - Epoch(test) [ 2/57]    eta: 0:03:23  time: 3.6978  data_time: 0.4468  memory: 1112  
08/30 12:29:21 - mmengine - INFO - Epoch(test) [ 3/57]    eta: 0:02:17  time: 2.5427  data_time: 0.3094  memory: 1112  
08/30 12:29:21 - mmengine - INFO - Epoch(test) [ 4/57]    eta: 0:01:45  time: 1.9965  data_time: 0.2535  memory: 1112  
08/30 12:29:22 - mmengine - INFO - Epoch(test) [ 5/57]    eta: 0:01:26  time: 1.6639  data_time: 0.2118  memory: 1112  
08/30 12:29:22 - mmengine - INFO - Epoch(test) [ 6/57]    eta: 0:01:12  time: 1.4271  data_time: 0.1832  memory: 1112  
08/30 12:29:22 - mmengine - INFO - Epoch(test) [ 7/57]    eta: 0:01:02  time: 1.2572  data_time: 0.1620  memory: 1112  
08/30 12:29:23 - mmengine - INFO - Epoch(test) [ 8/57]    eta: 0:00:56  time: 1.1508  data_time: 0.1462  memory: 1112  
08/30 12:29:23 - mmengine - INFO - Epoch(test) [ 9/57]    eta: 0:00:50  time: 1.0600  data_time: 0.1336  memory: 1112  
08/30 12:29:23 - mmengine - INFO - Epoch

In [6]:
output_test_data

{'coco/bbox_mAP': 0.805,
 'coco/bbox_mAP_50': -1.0,
 'coco/bbox_mAP_75': -1.0,
 'coco/bbox_mAP_s': 0.805,
 'coco/bbox_mAP_m': -1.0,
 'coco/bbox_mAP_l': -1.0,
 'coco/bbox_AR@100': 0.851}

# Cycle Loop

The maximum severity is 0.3. 

In [10]:
Results = {'Band':[], 'Severity':[], 'Precision':[], 'Recall':[], 'F1':[]}

for band_sel in range(1, 13, 1):
    for S in np.arange(0, .3, 0.025):
        try:
            # TODO: Update weight path according to band_selected.
            weight_path = get_weight_b(band_sel)
            cfg = set_NoiseTesting_config(weight_path, severity=S, corruption='gaussian', Seed=18)

            # Run the test
            runner = RUNNERS.build(cfg)
            work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
            runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
            output_test_data = runner.test()
        
        except Exception as e:
            print(f'Error in band {band_sel} and severity {S}')
            print(f'Trying with another seed..')
            
            cfg = set_NoiseTesting_config(weight_path, severity=S, corruption='gaussian', Seed=18)
            # Run the test
            runner = RUNNERS.build(cfg)
            work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
            runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
            output_test_data = runner.test()

        
        P = output_test_data['coco/bbox_mAP']
        R = output_test_data['coco/bbox_AR@100']
        
        Results['Band'].append(band_sel)
        Results['Severity'].append(S)
        Results['Precision'].append(P)
        Results['Recall'].append(R)
        F1 = 2 * (P * R) / (P + R)
        Results['F1'].append(F1)

Loading from: /Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b1/18_BS_3_LR_0.0008_ME_30_OPT_SGD/epoch_30.pth
08/30 12:33:16 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 18
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda-11.4
    NVCC: Cuda compilation tools, release 11.4, V11.4.152
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usu

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


08/30 12:33:17 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
08/30 12:33:17 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


08/30 12:34:00 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
08/30 12:34:00 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

...

08/30 13:37:06 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
08/30 13:37:06 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


08/30 13:37:40 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
08/30 13:37:40 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


08/30 13:38:13 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
08/30 13:38:13 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

In [ ]:
pd.DataFrame(Results).to_pickle('/Data_large/marine/PythonProjects/MMDET/notebooks/plotters/VENuS/TTA_SpikeNoise.pkl')

# Plot

In [ ]:
from style import set_style
from matplotlib import pyplot as plt

set_style(scale_factor=1.5, fontsize=9)


df = pd.DataFrame(Results)


def get_metrics_x_band(df, band):
    df = df[df['Band'] == band]
    return df


b1 = get_metrics_x_band(df, 1)


fig = plt.figure()
for idx, band in enumerate(range(1, 13, 1)):
    
    plt.subplot(2,6,band)

    plt.title('($B_{'+str(band)+'}$)', x=0.1, y=0.85, fontdict={'fontsize': 18})
    
    b = get_metrics_x_band(df, band)
    x = b['Severity'].to_numpy()
    y1 = b['Precision'].to_numpy()
    y2 = b['Recall'].to_numpy()
    y3 = b['F1'].to_numpy()
    
    plt.plot(x, y1, label=f'$Precision$')
    plt.plot(x, y2, label=f'$Recall$')
    plt.plot(x, y3, label=f'$F-1 Score$')
    
    if idx in [0, 6]:
        plt.ylabel('Metric Value')
    
    if idx in [6, 7, 8, 9, 10, 11]:
        plt.xlabel('Severity [$\sigma$]')
    
    plt.ylim(0, 1)
    plt.xlim(0, 5)
    
    plt.legend()

plt.tight_layout()
plt.savefig('/Data_large/marine/PythonProjects/MMDET/plots/SpikeVenus.png')
plt.show()
